# caudal, a walkthrough

*Reservoir inflow forecasting with a quantile LSTM, scored in public against the official ensemble.*

This notebook walks the whole thing: the idea, the points and their observations, the forcing, how the training
frames are built, the model and its loss, what it scores on ten held-out water years (all days and atmospheric-river
days), how calibrated it is, how it runs every day into the public ledger, and what the paper could claim.
Every number here is recomputed from the data on disk and the saved checkpoint, so it stays honest when either changes.

**The idea in one paragraph.** California's reservoir operators and the river forecast center (CNRFC) issue daily
inflow forecasts; the CNRFC's HEFS ensemble is the operational reference, and it is only published live, never
archived. `hapi` (public) archives it every six hours, fetches the observed inflow from CDEC, and scores
everything that is dropped into it. `caudal` (this repo, private) is the model that drops forecasts in. Because every
forecast is committed before its outcome exists, the record that accumulates is one nobody can fake: a model either
beats the official product prospectively, or it does not.


In [ ]:
import json, sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

from caudal import forcing, frames as F
from caudal.config import DATA_ROOT, LEDGER_DATA, MODELS_DIR, RESULTS_DIR
from caudal.data.cw3e import basin_ar_frame
from caudal.evaluate import load_model
from caudal.models.quantile_lstm import DEFAULT_QUANTILES, predict_quantiles
from caudal.points import POINTS, REGULATION, TRAINED
from caudal.scores import crps_ensemble

INK, INK2, GRID = "#0b0b0b", "#52514e", "#e8e7e3"
BLUE, ORANGE, AQUA, VIOLET = "#2a78d6", "#eb6834", "#1baf7a", "#6250d6"
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.titleweight": "medium", "axes.titlelocation": "left",
                     "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
                     "grid.color": GRID, "grid.linewidth": 0.8, "xtick.major.size": 0, "ytick.major.size": 0,
                     "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white", "figure.dpi": 110})
flow_fmt = FuncFormatter(lambda v, _: f"{v / 1000:g}K" if v >= 10_000 else f"{v:,.0f}")
RUN = "quantile_inflow_v1_s0"
print("data:", DATA_ROOT, "| ledger:", LEDGER_DATA)


## 1. The points

Fifteen reservoirs whose inflow CNRFC forecasts with HEFS and whose observed inflow CDEC publishes (sensor 76,
daily mean, cfs). Inflow is the right target for a regulated river: upstream of the dam the basin behaves as a
natural catchment, so a rainfall-runoff model applies without knowing release rules. Four stations compute inflow
from storage change and release and go negative at low flow; they are archived but not scored or trained on.
Each point carries a regulation label that says what the model is asked to predict; the flood gauges on the North
Coast rivers are in the ledger too (HEFS and the NWS official forecast archived) but not yet observed or scored.


In [ ]:
pts = pd.DataFrame([(lid, *v, REGULATION[lid], lid in TRAINED) for lid, v in POINTS.items()],
                   columns=["lid", "cdec", "name", "lat", "lon", "dam_ft", "regulation", "trained"]).set_index("lid")
obs = {lid: F.load_obs(lid) for lid in POINTS}
raw = {lid: pd.read_csv(LEDGER_DATA / "cdec_inflow" / f"{POINTS[lid][0]}.csv", index_col="date", parse_dates=True)["cfs"] for lid in POINTS}
pts["record_start"] = [raw[l].index[0].date() for l in pts.index]
pts["days"] = [len(raw[l]) for l in pts.index]
pts["negative_days"] = [(raw[l] < 0).sum() for l in pts.index]
pts["mean_cfs"] = [round(obs[l].mean()) for l in pts.index]
pts.sort_values("trained", ascending=False)


In [ ]:
# The annual cycle of inflow, every trained point: the Sierra snowmelt reservoirs peak in May-June, the
# coastal and northern ones in winter. The model must learn both from the same weights.
cyc = pd.DataFrame({POINTS[l][1]: obs[l].groupby(obs[l].index.month).mean() for l in TRAINED})
fig, ax = plt.subplots(figsize=(11, 4.2))
norm = cyc / cyc.max()
im = ax.imshow(norm.T, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax.set_yticks(range(len(norm.columns)), norm.columns); ax.set_xticks(range(12), "JFMAMJJASOND")
ax.grid(False); ax.set_title("Mean monthly inflow, each reservoir scaled to its own peak month")
for i, name in enumerate(norm.columns):
    ax.text(12.2, i, f"peak {cyc[name].max():,.0f} cfs", va="center", fontsize=8, color=INK2)
fig.tight_layout()


## 2. The forcing

Open-Meteo serves ERA5 (history, about five days behind real time) and the forecast models (next 16 days) with the
same daily variables: precipitation, max, min and mean temperature, FAO reference evapotranspiration. Training on the
archive and serving on the forecast keeps the model's inputs consistent, the "train = serve forcing" rule. The
shortcut for now: forcing is sampled at the dam, not averaged over the basin, which hurts the high Sierra reservoirs
most (their rain falls as snow a thousand metres above the dam). Basin-mean forcing over the HUC polygons is the first
upgrade once the live record exists.

Water year 2017 at Oroville shows what the model has to learn: a sequence of atmospheric rivers in January and
February, the inflow response within a day, and the snowmelt plateau through May.


In [ ]:
lid = "ORDC1"
f = forcing.load(lid).loc["2016-10-01":"2017-09-30"]
q = obs[lid].loc["2016-10-01":"2017-09-30"]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True, height_ratios=[1, 1.6])
a1.bar(f.index, f.precip, width=1, color=BLUE, lw=0); a1.set_ylabel("precip at the dam, mm/d")
a1.set_title(f"{POINTS[lid][1]}: water year 2017")
a2.plot(q.index, q.values, color=INK, lw=1.5); a2.set_yscale("log"); a2.yaxis.set_major_formatter(flow_fmt)
a2.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5))); a2.yaxis.set_minor_formatter(NullFormatter())
a2.set_ylabel("observed inflow, cfs (log)")
ar = basin_ar_frame(POINTS[lid][2], "2016-10-01", "2017-09-30")
for d in ar.index[ar.is_ar_day]:
    a2.axvspan(d, d + pd.Timedelta(days=1), color=ORANGE, alpha=0.18, lw=0)
a2.text(pd.Timestamp("2017-06-15"), q.max() * 0.8, "orange: AR days (CW3E catalog, scale 3+)", color=ORANGE, fontsize=8)
a2.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
fig.tight_layout()


## 3. From data to training frames

`caudal.frames` follows caudal-fleet's regional protocol, the one that produced the paper's 71-basin fleet:

- **One model for all points.** Features are standardized with statistics pooled over every point's fit window;
  the target is scaled per point by its own fit-window standard deviation so Shasta does not drown Success in the loss.
- **Dynamic features** per day: precip, tmax, tmin, tmean, pet, and the day of year as sine and cosine.
  **Static features** per point: latitude, dam elevation, log of mean inflow. Three numbers that let one network
  condition on which basin it is looking at.
- **The one-day shift.** The window that predicts day *d* ends with day *d*'s own forcing, which a forecast run has
  in hand (as forecast). Streamflow is never an input, so the model is purely forcing-driven and can be run as far
  ahead as the weather forecast goes.
- **Split in time, never at random.** Fit through 2015-09-30, evaluate water years 2016-2025. Daily flow is
  autocorrelated, so random holdout would leak.


In [ ]:
model, ckpt = load_model(RUN)
frames = F.build_regional(ckpt["points"], ckpt["fit_end"])
ex = frames["raw"]["SHDC1"].loc["2017-02-05":"2017-02-12", F.FEATURES + [F.TARGET]]
print(f"features ({len(F.FEATURES)}):", F.FEATURES)
print(f"lookback {F.LOOKBACK} d, hidden {F.HIDDEN}, batch {F.BATCH_SIZE}, fit through {F.FIT_END}, eval {F.EVAL_START} to {F.EVAL_END}")
print("training windows:", f"{ckpt['n_windows']:,}", "| target scale per point (fit-window std, cfs):",
      {k: round(v) for k, v in list(frames["s_target"].items())[:4]}, "...")
ex.round(2)


## 4. The model

`QuantileLSTM` (from caudal-fleet): a single-layer LSTM over the 365-day window, and a head that emits the lowest quantile
plus non-negative softplus increments, so the seven quantiles (0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99) can never
cross by construction. The loss is the mean **pinball loss** over the grid,
$\ell_q(y, \hat y) = \max\big(q\,(y-\hat y),\,(q-1)(y-\hat y)\big)$,
which penalises under-prediction by $q$ and over-prediction by $1-q$: the high-$q$ heads are pushed upward,
exactly the asymmetry a Gaussian mean lacks at flood peaks. Each head is a consistent estimator of its own quantile
with no distributional assumption.

Why quantiles rather than a Gaussian head: caudal-fleet's finding was that the Gaussian LSTM's mean under-predicts 98% of
AR flood peaks while its own upper quantile recovers them; training for the tail directly removes the selection step.
Learning rate 1e-3 matters: higher rates saturate the LSTM and the heads collapse together while the loss still looks
fine, so the trainer checks the predicted spread and refuses to save a collapsed run.


In [ ]:
m = pd.read_csv(RESULTS_DIR / "runs" / RUN / "metrics.csv")
cfg = json.load(open(RESULTS_DIR / "runs" / RUN / "config.json"))
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(m.epoch, m.train_pinball, color=BLUE, lw=2, marker="o", ms=3)
ax.set(xlabel="epoch", ylabel="mean pinball loss (scaled target)", title=f"{RUN}: {cfg['config']['epochs']} epochs, "
       f"{cfg['environment'].get('platform', '')[:20]}")
fig.tight_layout()
print({k: cfg["config"][k] for k in ["lookback", "hidden", "lr", "epochs", "seed", "n_windows"]})


## 5. Retrospective skill, water years 2016-2025

Scores on the held-out decade, from `caudal.evaluate`. CRPS in cfs over the seven quantiles treated as an ensemble
(lower is better); persistence is yesterday's observed inflow, the hardest lead-1 reference there is; climatology is the
fit-window ensemble for that day of year. **AR days** are days a landfalling atmospheric river of scale 3 or more (the
CW3E catalog, Ralph et al. 2019) touched the coast within 1.5 degrees of the dam's latitude, caudal-fleet's definition.

Caveat that matters for the paper: these are simulation scores, ERA5 through the target day. Live forecasts use
forecast forcing for leads 1-14, so live skill will be lower, and that live number is what the ledger measures.


In [ ]:
ev = pd.read_csv(RESULTS_DIR / f"eval_{RUN}.csv", index_col="lid")
ev = ev[ev.lead == 0].drop(columns="lead") if "lead" in ev else ev  # tables written since v2 hold one row per lead
ev.insert(0, "name", [POINTS[l][1] for l in ev.index])
show = ev[["name", "n_days", "n_ar_days", "crps_model", "crps_persistence", "crps_climatology",
           "crps_model_ar", "crps_persistence_ar", "crps_climatology_ar", "cov90", "nse_median"]]
pooled = show.drop(columns=["name", "n_days", "n_ar_days"]).mean()
pd.concat([show, pooled.to_frame("pooled").T]).round(2)


In [ ]:
# Skill against persistence, 1 - CRPS_model / CRPS_persistence: above zero the model beats yesterday's value.
sk = pd.DataFrame({"all days": (1 - ev.crps_model / ev.crps_persistence).values,
                   "AR days": (1 - ev.crps_model_ar / ev.crps_persistence_ar).values}, index=ev.name)
fig, ax = plt.subplots(figsize=(10, 3.8))
x = np.arange(len(sk)); w = 0.38
ax.bar(x - w / 2, sk["all days"], w, color=GRID, label="all days")
ax.bar(x + w / 2, sk["AR days"], w, color=ORANGE, label="AR days")
ax.axhline(0, color=INK2, lw=0.8); ax.set_xticks(x, sk.index, rotation=25, ha="right")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:+.0%}"))
ax.set_title("Skill vs persistence (CRPS), held-out water years 2016-2025: the skill is concentrated on AR days")
ax.legend(loc="upper left", ncol=2)
fig.tight_layout()


Read: on all days the model roughly ties persistence, which is what a lead-1 simulation of a slow-moving
quantity should do. On AR days, when the inflow is about to change by a factor of ten, persistence is useless and the
model keeps most of its accuracy. That is the sentence the paper is built on, and the ledger will test it with
forecast forcing.

## 6. What a forecast looks like through an event

Run the saved model over the January-February 2017 sequence at Oroville, the event that led to the spillway
failure. The band is the 5-95% range, the line the median, the dark line what CDEC observed.


In [ ]:
def predict_point(lid, start, end):
    r, s = frames["raw"][lid], frames["s_target"][lid]
    hist = r.loc[:start].iloc[-(F.LOOKBACK + 1):-1]
    z = F.prep(pd.concat([hist, r.loc[start:end]]), frames["f_mean"], frames["f_std"], s)
    preds, obs_s, idx = predict_quantiles(model, z, F.FEATURES, F.TARGET, F.LOOKBACK)
    out = pd.DataFrame(np.clip(preds * s, 0, None), index=idx, columns=[f"q{int(round(q * 100)):02d}" for q in DEFAULT_QUANTILES])
    out["obs"] = obs_s * s
    return out

ev_pred = {lid: predict_point(lid, F.EVAL_START, F.EVAL_END) for lid in TRAINED}
p = predict_point("ORDC1", "2016-12-15", "2017-03-15")
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.fill_between(p.index, p.q05, p.q95, color=VIOLET, alpha=0.15, lw=0, label="model 5-95%")
ax.fill_between(p.index, p.q25, p.q75, color=VIOLET, alpha=0.3, lw=0, label="model 25-75%")
ax.plot(p.index, p.q50, color=VIOLET, lw=2, label="model median")
ax.plot(p.index, p.obs, color=INK, lw=2, label="observed")
ax.set_yscale("log"); ax.yaxis.set_major_formatter(flow_fmt); ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
ax.yaxis.set_minor_formatter(NullFormatter()); ax.set_ylabel("inflow, cfs (log)")
ax.set_title("Oroville, winter 2017: the saved model driven by ERA5 (a simulation, not a forecast)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d")); ax.legend(ncol=4, loc="upper left")
fig.tight_layout()


## 7. Calibration

A quantile forecast is calibrated when the observation falls below the $q$ quantile a fraction $q$ of the time.
Pooled over the eleven points and the decade, the curve below should sit on the diagonal. It does not: the 90% band
covers about 60% of days, the same under-dispersion caudal-fleet found for this head. The fix is **split conformal quantile
regression** (Romano et al. 2019): widen each quantile by the offset that achieves nominal coverage on a calibration
window the optimizer never saw. It is a hundred lines, it is the paper's one new computation, and the ledger is where
its live coverage gets checked every day.


In [ ]:
allp = pd.concat(ev_pred.values())
qcols = [c for c in allp.columns if c.startswith("q")]
nominal = np.array([int(c[1:]) / 100 for c in qcols])
below = np.array([(allp.obs <= allp[c]).mean() for c in qcols])
# AR days only, with the same pooled definition the evaluator uses
ar_mask = pd.concat([basin_ar_frame(POINTS[l][2], F.EVAL_START, F.EVAL_END)["is_ar_day"].reindex(ev_pred[l].index).fillna(False) for l in TRAINED])
below_ar = np.array([(allp.obs[ar_mask.values] <= allp[c][ar_mask.values]).mean() for c in qcols])
fig, ax = plt.subplots(figsize=(4.8, 4.4))
ax.plot([0, 1], [0, 1], color=GRID, lw=1); ax.plot(nominal, below, "o-", color=BLUE, lw=2, label="all days")
ax.plot(nominal, below_ar, "o-", color=ORANGE, lw=2, label="AR days")
ax.set(xlabel="nominal quantile", ylabel="fraction of observations below", title="Reliability of the quantiles")
ax.legend(loc="upper left"); fig.tight_layout()
print("coverage of the 5-95% band:", f"all days {(below[-2] - below[0]):.2f}, AR days {(below_ar[-2] - below_ar[0]):.2f}")


## 8. The operational loop and the first day of the record

Every day `scripts/issue.py` extends the ERA5 archive, bridges to today with the forecast model's analysis days,
appends the 16-day forecast, runs the model and writes fourteen days of quantiles per point into
`hapi/data/forecasts/<model>/<lid>/<date>.csv`. The ledger's `score.py` scores that file the same way it
scores HEFS the moment CDEC publishes each day's inflow. Below, the first issued forecast beside the HEFS ensemble
archived the same day. Note Shasta's last lead: the GFS run put 164 mm on the basin on day 14 and the model, driven by
that single deterministic trajectory, follows it, while HEFS spreads its members. That overconfidence at long leads is
v1's known weakness, and forecast-ensemble forcing is the layer that addresses it.


In [ ]:
import gzip

def hefs_fan(lid):
    p = sorted((LEDGER_DATA / "hefs_daily" / lid).glob("*.gz"))[-1]
    lines = gzip.decompress(p.read_bytes()).decode().splitlines()[2:]
    rows = {pd.Timestamp(l[:10]): np.array([float(x) for x in l.split(",")[1:] if x]) * 1000 for l in lines}
    days = sorted(rows)[:16]
    return pd.DataFrame({d: np.percentile(rows[d], [5, 25, 50, 75, 95]) for d in days}, index=["p05", "p25", "p50", "p75", "p95"]).T

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, lid in zip(axes, ["SHDC1", "ORDC1", "FOLC1"]):
    fc = sorted((LEDGER_DATA / "forecasts" / RUN / lid).glob("*.csv"))[-1]
    mdl = pd.read_csv(fc, index_col="date", parse_dates=True)
    h = hefs_fan(lid)
    o = obs[lid].loc[h.index[0] - pd.Timedelta(days=30):]
    ax.fill_between(h.index, h.p05, h.p95, color=BLUE, alpha=0.15, lw=0); ax.plot(h.index, h.p50, color=BLUE, lw=2, label="HEFS median")
    ax.fill_between(mdl.index, mdl.q05, mdl.q95, color=VIOLET, alpha=0.15, lw=0); ax.plot(mdl.index, mdl.q50, color=VIOLET, lw=2, label="caudal v1 median")
    ax.plot(o.index, o.values, color=INK, lw=2, label="observed")
    ax.set_yscale("log"); ax.yaxis.set_major_formatter(flow_fmt); ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_title(f"{POINTS[lid][1]}, issued {fc.stem}"); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d")); ax.tick_params(axis="x", labelrotation=30)
axes[0].legend(fontsize=8); fig.tight_layout()


## 9. The paper

**Working claim.** *During atmospheric-river events, a learned quantile model of reservoir inflow carries skill that
the operational ensemble and simple references do not, and a public prospective record shows it.* Three results
support it, two already in hand retrospectively and one that only time produces:

1. **AR-conditional skill.** Skill against persistence and climatology, all days versus AR days, over ten water years
   and eleven reservoirs (section 5). The novelty is the conditioning: hydrologic ML papers report pooled NSE and
   CRPS; flood managers care about the days in the orange band. caudal-fleet already showed the pooled-vs-AR gap on 71
   natural basins; this moves it to the regulated, decision-relevant quantity, reservoir inflow, where the operator's
   flood-control release depends on the forecast.
2. **Calibration, before and after conformal.** Section 7's curve, then the same curve after split conformal
   calibration fit on 2010-2015, reported all-day and AR-day. The method is standard; the finding that interests a
   reviewer is whether AR-day calibration survives a correction fit mostly on dry days (the hypothesis: it does not,
   and an AR-conditional conformal offset is needed).
3. **The prospective record.** Live CRPS, coverage and AR-event peak error of the model, HEFS, persistence and
   climatology, scored day by day from the public ledger with forecast forcing. Even a short winter gives a few
   events; an honest paper reports the all-day record and says the event sample is pending. This is the part that
   separates the work from a benchmark study: forecasts timestamped before the outcome, scored against the agency's
   own product.

**Secondary threads, each a dated layer in the ledger rather than a promise.**

- *Forcing uncertainty.* v1 runs one deterministic GFS trajectory and is overconfident at long leads (section 8).
  Running the model over an ensemble of forcings (GEFS, or Open-Meteo's multi-model set) separates forcing
  uncertainty from model uncertainty; HEFS does this by construction, so the comparison is fair only after this layer.
- *Observed-flow handoff.* Feeding the last seven observed days to the model (v2) tests whether gauges fix the slow
  start on flood gauges below dams, the regulation treatment the points registry already names.
- *Operating quantile.* caudal-fleet selected the flood operating quantile on the training window; here the same choice can
  be scored live: which quantile would an operator have used to decide releases, and how often was it right.
- *Basin-mean forcing and statics.* The honest ablation: dam-point forcing versus basin averages, and whether three
  static numbers suffice for a regional model of eleven basins.

**Venue and shape.** HESS or Journal of Hydrometeorology, short paper: Methods (data, protocol, the head and loss,
conformal, AR definition), Results (the three claims), a Prospective section written last from the ledger, and a Data
and Code statement pointing at the public repository and its data branch, which is the archive itself.

**What is missing before writing.** Conformal calibration (one script, one figure); the forcing-ensemble layer or an
explicit statement of its absence; the flood gauges' observed flow in the ledger (USGS daily values) so the North Coast
rivers are scored too; and at least sixty days of live record.
